# Filtrar imagens com anotações `inside-macrophage`

Este notebook lê o dataset no formato CVAT/YOLO e retorna somente as imagens
que possuem pelo menos uma anotação da classe `inside-macrophage`.

- Autor: Clésio Gonçalves
- Data: 10 abr. 2026

## Importações e parâmetros

In [1]:
import os
import shutil
import pandas as pd

# ── Ajuste estes parâmetros conforme seu ambiente ──────────────────────────
diretorio_cvat   = 'dataset_cvat'       # pasta raiz do dataset CVAT
subpasta_objetos = 'obj_train_data'     # subpasta com imagens e .txt
extensao         = '.png'               # extensão das imagens
classe_alvo      = 'inside-macrophage'  # classe que queremos filtrar
diretorio_saida  = 'dataset_inside_macrophage'  # onde salvar o resultado
# ───────────────────────────────────────────────────────────────────────────

## 1 – Descobrir o índice da classe `inside-macrophage` no `obj.names`

O arquivo `obj.names` lista uma classe por linha. O índice (base 0) é o mesmo
número usado nos arquivos `.txt` de anotação do YOLO.

In [2]:
# Procura o obj.names em qualquer lâmina e retorna o índice da classe.
def encontra_indice_classe(diretorio_cvat, classe_alvo):
    for lamina in os.listdir(diretorio_cvat):
        obj_names = os.path.join(diretorio_cvat, lamina, 'obj.names')
        if os.path.isfile(obj_names):
            with open(obj_names, 'r') as f:
                classes = [linha.strip() for linha in f.readlines()]
            print(f"Classes encontradas em '{obj_names}':")
            for i, c in enumerate(classes):
                marker = '  < alvo' if c == classe_alvo else ''
                print(f"  {i}: {c}{marker}")
            if classe_alvo in classes:
                return classes.index(classe_alvo)
            else:
                raise ValueError(f"Classe '{classe_alvo}' não encontrada em {obj_names}.")
    raise FileNotFoundError("Nenhum arquivo obj.names encontrado no dataset.")

indice_alvo = encontra_indice_classe(diretorio_cvat, classe_alvo)
print(f"\nÍndice da classe '{classe_alvo}': {indice_alvo}")

Classes encontradas em 'dataset_cvat/leish_5_dataset_2026_03_18_10_24_35_yolo 1.1.zip/obj.names':
  0: Non-macrophages
  1: macrophage
  2: leishmania
  3: inside-macrophage  < alvo

Índice da classe 'inside-macrophage': 3


## 2 – Varrer todas as imagens e manter as que têm anotação da classe alvo

In [3]:
def tem_classe(txt_path, indice_alvo):
    """Retorna True se o .txt de anotação YOLO contiver a classe alvo."""
    if not os.path.isfile(txt_path):
        return False
    with open(txt_path, 'r') as f:
        for linha in f:
            linha = linha.strip()
            if not linha:
                continue
            classe_id = int(linha.split()[0])
            if classe_id == indice_alvo:
                return True
    return False


def filtra_imagens(diretorio_cvat, subpasta_objetos, extensao, indice_alvo):
    """Retorna DataFrame apenas com imagens que contêm a classe alvo."""
    registros = []

    for lamina in sorted(os.listdir(diretorio_cvat)):
        diretorio = os.path.join(diretorio_cvat, lamina, subpasta_objetos)
        if not os.path.isdir(diretorio):
            continue

        total = 0
        encontradas = 0
        for arquivo in sorted(os.listdir(diretorio)):
            if not arquivo.endswith(extensao):
                continue
            total += 1
            nome_base = os.path.splitext(arquivo)[0]
            txt_path  = os.path.join(diretorio, f"{nome_base}.txt")
            if tem_classe(txt_path, indice_alvo):
                encontradas += 1
                registros.append({
                    'diretorio': diretorio,
                    'imagem':    arquivo,
                    'lamina':    lamina
                })

        print(f"Lâmina '{lamina}': {encontradas}/{total} imagens com '{classe_alvo}'")

    return pd.DataFrame(registros)


dataset_filtrado = filtra_imagens(diretorio_cvat, subpasta_objetos, extensao, indice_alvo)
print(f"\nTotal de imagens com '{classe_alvo}': {len(dataset_filtrado)}")

Lâmina 'leish_1_dataset_2026_03_17_12_10_29_yolo 1.1.zip': 67/67 imagens com 'inside-macrophage'
Lâmina 'leish_2_dataset_2026_03_17_12_18_07_yolo 1.1.zip': 46/52 imagens com 'inside-macrophage'
Lâmina 'leish_3_dataset_2026_03_16_02_54_37_yolo 1.1.zip': 11/14 imagens com 'inside-macrophage'
Lâmina 'leish_4_dataset_2026_03_17_12_25_16_yolo 1.1.zip': 47/47 imagens com 'inside-macrophage'
Lâmina 'leish_5_dataset_2026_03_18_10_24_35_yolo 1.1.zip': 127/127 imagens com 'inside-macrophage'
Lâmina 'leish_7_dataset_2026_03_18_10_34_17_yolo 1.1.zip': 95/95 imagens com 'inside-macrophage'
Lâmina 'leish_8_dataset_2026_03_18_10_49_36_yolo 1.1.zip': 134/134 imagens com 'inside-macrophage'
Lâmina 'leish_9_dataset_2026_03_18_10_55_05_yolo 1.1.zip': 72/73 imagens com 'inside-macrophage'

Total de imagens com 'inside-macrophage': 599


## 3 – Visualizar o resultado

In [4]:
dataset_filtrado

,diretorio,imagem,lamina
0,dataset_cvat/leish_1_dataset_2026_03_17_12_10_...,leish_1_0.png,leish_1_dataset_2026_03_17_12_10_29_yolo 1.1.zip
1,dataset_cvat/leish_1_dataset_2026_03_17_12_10_...,leish_1_1.png,leish_1_dataset_2026_03_17_12_10_29_yolo 1.1.zip
2,dataset_cvat/leish_1_dataset_2026_03_17_12_10_...,leish_1_10.png,leish_1_dataset_2026_03_17_12_10_29_yolo 1.1.zip
3,dataset_cvat/leish_1_dataset_2026_03_17_12_10_...,leish_1_11.png,leish_1_dataset_2026_03_17_12_10_29_yolo 1.1.zip
4,dataset_cvat/leish_1_dataset_2026_03_17_12_10_...,leish_1_12.png,leish_1_dataset_2026_03_17_12_10_29_yolo 1.1.zip
...,...,...,...
594,dataset_cvat/leish_9_dataset_2026_03_18_10_55_...,Leish_09_72.png,leish_9_dataset_2026_03_18_10_55_05_yolo 1.1.zip
595,dataset_cvat/leish_9_dataset_2026_03_18_10_55_...,Leish_09_73.png,leish_9_dataset_2026_03_18_10_55_05_yolo 1.1.zip
596,dataset_cvat/leish_9_dataset_2026_03_18_10_55_...,Leish_09_75.png,leish_9_dataset_2026_03_18_10_55_05_yolo 1.1.zip
597,dataset_cvat/leish_9_dataset_2026_03_18_10_55_...,Leish_09_76.png,leish_9_dataset_2026_03_18_10_55_05_yolo 1.1.zip


In [5]:
print("Distribuição por lâmina:")
print(dataset_filtrado['lamina'].value_counts())

Distribuição por lâmina:
lamina
leish_8_dataset_2026_03_18_10_49_36_yolo 1.1.zip    134
leish_5_dataset_2026_03_18_10_24_35_yolo 1.1.zip    127
leish_7_dataset_2026_03_18_10_34_17_yolo 1.1.zip     95
leish_9_dataset_2026_03_18_10_55_05_yolo 1.1.zip     72
leish_1_dataset_2026_03_17_12_10_29_yolo 1.1.zip     67
leish_4_dataset_2026_03_17_12_25_16_yolo 1.1.zip     47
leish_2_dataset_2026_03_17_12_18_07_yolo 1.1.zip     46
leish_3_dataset_2026_03_16_02_54_37_yolo 1.1.zip     11
Name: count, dtype: int64


## 4 – Copiar imagens e `.txt` filtrados para uma pasta de saída

In [6]:
os.makedirs(diretorio_saida, exist_ok=True)

for _, row in dataset_filtrado.iterrows():
    nome_base = os.path.splitext(row['imagem'])[0]

    img_origem = os.path.join(row['diretorio'], row['imagem'])
    txt_origem = os.path.join(row['diretorio'], f"{nome_base}.txt")

    shutil.copy(img_origem, os.path.join(diretorio_saida, row['imagem']))
    if os.path.isfile(txt_origem):
        shutil.copy(txt_origem, os.path.join(diretorio_saida, f"{nome_base}.txt"))

print(f"{len(dataset_filtrado)} imagens (+ .txt) copiadas para '{diretorio_saida}'.")

599 imagens (+ .txt) copiadas para 'dataset_inside_macrophage'.
